# 第 6 周练习 — 微调（「价格合适」）

## 练习目标（理念）

根据商品描述**预测价格**：先整理亚马逊相关数据（`curate.py`，复用课程 `pricer` 包），测量**零样本（zero-shot）**前沿基线，再**微调（fine-tune）**一个本地小模型做对比。

## 为什么不用 OpenAI API 微调？

课程原演示通过 OpenAI API 微调 `gpt-4o-mini`，但 OpenAI **已弃用/收紧 API 微调**（常见报错 403 `training_not_available`）。Gemini 调优往往还要 Vertex/GCP，不只是一把 API Key。因此本练习改为微调**本地开源**模型（`distilgpt2` + CPU 上的 **LoRA**）——这也是在预习第 7 周常见技术。

## 怎么跑

1. 先在本目录跑：`curate.py ALL data_small.pkl 5000`（生成小规模 pickle）  
2. 准备好 `.env` 里的 OpenAI 密钥（零样本基线会用到）  
3. 从上到下运行：加载数据 → 零样本 MAE/命中率 → LoRA 训练 → 基座 vs 微调 vs 前沿对比  


In [1]:
# ========== 导入 + 环境 + 找到课程仓库根目录 ==========

# 标准库：sys（改模块搜索路径）、pickle（读数据）、re（抽价格数字）、random（打乱）
import sys, pickle, re, random
# Path：拼路径、向上查找仓库根
from pathlib import Path
# PyTorch：本地训练与推理的张量框架
import torch
# transformers：加载因果语言模型与分词器（Tokenizer）
from transformers import AutoModelForCausalLM, AutoTokenizer
# peft：LoRA 配置与把适配器挂到基座模型上
from peft import LoraConfig, get_peft_model
# OpenAI 客户端：后面做零样本 gpt-4o-mini 基线
from openai import OpenAI
# load_dotenv：把 .env 密钥读进环境变量
from dotenv import load_dotenv

# 加载 .env；override=True 允许覆盖已有环境变量
load_dotenv(override=True)
# 默认读取 OPENAI_API_KEY 等环境变量，构造云端客户端
openai = OpenAI()
# 从当前目录向上找：哪个祖先目录里有 week6/pricer/items.py，那个就是仓库根
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "week6/pricer/items.py").exists())
# 把 week6 放进 sys.path，才能 import pricer
sys.path.insert(0, str(REPO / "week6"))
# Item：课程里的商品数据模型（Pydantic）；noqa 忽略「导入不在文件顶部」的 lint
from pricer.items import Item  # noqa: E402
# 本练习数据所在目录（注意路径仍是原作者的 community-contributions/...）
HERE = REPO / "community-contributions/NicholasDean/week6"


In [2]:
# ========== 加载精简数据集 pickle ==========

# 读 data_small.pkl（混杂子集；完整策划版是 data_full.pkl，约 9GB）
blob = pickle.load(open(HERE / "data_small.pkl", "rb"))   # mixed subset (full curation = data_full.pkl, ~9GB)
# 把 dict 列表校验/还原成 Item 对象：训练集
train = [Item.model_validate(d) for d in blob["train"]]
# 测试集同理
test = [Item.model_validate(d) for d in blob["test"]]
# 评估时只用前 50 条，加快演示
SAMPLE = test[:50]
# 打印规模（千分位逗号）
print(f"train={len(train):,}  test={len(test):,}")


train=16,801  test=2,000


## 零样本基线（前沿参考）

用 `gpt-4o-mini` 直接估价，**不做训练**。  
命中（hit）定义：绝对误差 ≤ 真实价格的 20%，或 ≤ 40 美元。


In [3]:
# ========== 零样本：用 gpt-4o-mini 估价格并算 MAE / 命中率 ==========

# system 提示：只让模型回价格格式（字符串必须原样保留）
SYSTEM = "You estimate the price of products. Reply with just the price, e.g. 'Price is $9.00'."

# 从模型回复文本里抽出第一个数字当价格；抽不到则 0.0
def get_price(text):
    # 去掉千分位逗号后再用正则找数字
    nums = re.findall(r"[-+]?\d*\.?\d+", text.replace(",", ""))   # first number in the text
    return float(nums[0]) if nums else 0.0

# 给定误差列表与对应 Item：返回平均绝对误差（MAE）与命中率
def score(errs, items):
    # 命中：误差 ≤ 20% 真价 或 ≤ 40 美元
    hits = sum(e <= 0.2 * it.price or e <= 40 for e, it in zip(errs, items))
    return sum(errs) / len(errs), hits / len(items)

# 对单个 Item：调 gpt-4o-mini，再解析回复里的价格
def gpt_price(item):
    r = openai.chat.completions.create(model="gpt-4o-mini", seed=42, max_tokens=8,
        messages=[{"role": "system", "content": SYSTEM}, {"role": "user", "content": item.test_prompt()}])
    return get_price(r.choices[0].message.content)

# 在 SAMPLE 上算零样本误差与命中率
mae0, hit0 = score([abs(gpt_price(it) - it.price) for it in SAMPLE], SAMPLE)
print(f"zero-shot gpt-4o-mini:  MAE=${mae0:,.2f}  hit-rate={hit0:.0%}")


zero-shot gpt-4o-mini:  MAE=$22.80  hit-rate=86%


## 使用 LoRA 微调 distilgpt2（本地，CPU）

**LoRA** 只训练很小的适配器权重（大约占总参数的 0.2%），因此在 CPU 上也相对便宜。  
训练样本来自策划好的完整 `prompt`（每条都以类似 `Price is $X.00` 的价格后缀结尾）。  
`truncation_side = "left"`：**左截断**，尽量保住提示词**末尾**的价格目标。


In [4]:
# ========== LoRA 微调 distilgpt2：两轮朴素训练循环 ==========

# 加载 distilgpt2 的分词器
tok = AutoTokenizer.from_pretrained("distilgpt2")
# GPT-2 家族默认没有 pad；用 eos 顶替，方便批处理/生成
tok.pad_token = tok.eos_token
# 左截断：超长时丢掉开头，保留结尾（价格目标）
tok.truncation_side = "left"                       # keep the END of the prompt (the price target)
# 基座因果 LM + LoRA：改 c_attn；任务类型为因果语言建模
model = get_peft_model(AutoModelForCausalLM.from_pretrained("distilgpt2"),
                       LoraConfig(r=8, lora_alpha=16, target_modules=["c_attn"], task_type="CAUSAL_LM"))
# 打印可训练参数占比，确认 LoRA 挂上了
model.print_trainable_parameters()

# 只用训练集前 150 条的完整训练 prompt（含答案后缀）
texts = [it.prompt for it in train[:150]]          # full training prompts
# AdamW 优化器，学习率 2e-4
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
# 进入训练模式（dropout 等开启）
model.train()
# 跑 2 个 epoch
for epoch in range(2):
    # 每个 epoch 用不同种子打乱，避免顺序固定
    random.Random(epoch).shuffle(texts)
    for txt in texts:
        # 分词并截断到 220；返回 PyTorch 张量
        enc = tok(txt, return_tensors="pt", truncation=True, max_length=220)
        # 因果 LM：labels 等于 input_ids，让模型预测下一个 token
        enc["labels"] = enc["input_ids"]
        # 前向算 loss 后反向传播
        model(**enc).loss.backward()
        # 更新参数，并清零梯度（同一行两个语句，逻辑保持原样）
        opt.step(); opt.zero_grad()
    print(f"epoch {epoch + 1} done")


c:\Users\Nicholas Dean\projects\llm_engineering\.venv\Lib\site-packages\peft\tuners\lora\layer.py:2504: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


trainable params: 147,456 || all params: 82,060,032 || trainable%: 0.1797


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


epoch 1 done


epoch 2 done


## 比较：基座 distilgpt2 vs 微调 vs 前沿模型


In [5]:
# ========== 本地生成估价 + 对比基座 / 微调 / 前沿 ==========

# 对单个 Item：用当前 model 生成短续写，再解析出价格数字
def local_price(item):
    # test_prompt：推理用提示（通常不含真价答案）
    enc = tok(item.test_prompt(), return_tensors="pt", truncation=True, max_length=220)
    # 推理时不建计算图，省显存/内存
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=8, do_sample=False, pad_token_id=tok.eos_token_id)
    # 只解码「新生成」的那一段 token，再抽价格
    return get_price(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))

# 批量评估：返回 MAE 与命中率
def eval_local(items):
    return score([abs(local_price(it) - it.price) for it in items], items)

# 评估模式（关闭 dropout）
model.eval()
# 适配器开启：微调后的效果
mae_ft, hit_ft = eval_local(SAMPLE)                      # adapters ON (fine-tuned)
# 临时关掉 LoRA 适配器：回到基座 distilgpt2
with model.disable_adapter():
    mae_base, hit_base = eval_local(SAMPLE)              # adapters OFF (base distilgpt2)

# 三路对比打印（含前面算过的 gpt-4o-mini 零样本）
print(f"base distilgpt2:       MAE=${mae_base:,.2f}  hit-rate={hit_base:.0%}")
print(f"fine-tuned distilgpt2: MAE=${mae_ft:,.2f}  hit-rate={hit_ft:.0%}")
print(f"gpt-4o-mini zero-shot: MAE=${mae0:,.2f}  hit-rate={hit0:.0%}  (frontier reference)")


base distilgpt2:       MAE=$97.36  hit-rate=70%
fine-tuned distilgpt2: MAE=$52.78  hit-rate=76%
gpt-4o-mini zero-shot: MAE=$22.80  hit-rate=86%  (frontier reference)
